# VKOSPI robust dynamic 개선 전략 — Colab 재현

VKOSPI를 한국판 VIX로 해석해 **장·단기 상대수준, robust z-score, 변동성 정규화 충격, 가속·고점거리**를 만들고, 2017년 이전 자료만으로 810개 일별 방어 오버레이를 다시 선택합니다.

- 함께 제공된 `vkospi_robust_dynamic_colab_bundle.zip`을 업로드하세요.
- 기본 실행은 약 4–8분이며 런타임에 따라 달라질 수 있습니다.
- 2018–2026은 선택에 쓰지 않는 잠금 검증입니다.
- 연구용 과거 시뮬레이션이며 투자 조언이 아닙니다.


## 1. 런타임과 번들 준비

In [ ]:
import sys, subprocess, json, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "numpy", "pandas", "matplotlib", "scipy", "scikit-learn", "openpyxl",
        ],
        check=True,
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def safe_extract(zip_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(destination)

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    bundle = next((Path(name) for name in uploaded if name.endswith(".zip")), None)
    if bundle is None:
        raise FileNotFoundError("vkospi_robust_dynamic_colab_bundle.zip을 업로드하세요.")
    safe_extract(bundle, Path("/content"))
    PROJECT_ROOT = Path("/content/RegimeDecisionTest")
else:
    PROJECT_ROOT = Path.cwd().resolve()

required = [
    "vkospi_dynamic_risk_experiment.py",
    "vkospi_robust_dynamic_experiment.py",
    "vkospi_extended_diagnostics.py",
    "vkospi_model_robustness.py",
    "regime_research.py",
    "raw_data/VKOSPIData.csv",
    "raw_data/compass.db",
    "raw_data/krx_bond_index.csv",
    "cache/market_daily.csv",
    "results/vkospi_selected_backtest.csv",
    "results/vkospi_dynamic_validation.json",
]
missing = [name for name in required if not (PROJECT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(missing)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("Python", sys.version.split()[0], "| 입력 확인 완료")


## 2. 810개 후보 재탐색

`RUN_SEARCH=True`가 기본입니다. 선정에는 2007–2017 전체와 2013–2017 내부검증만 쓰며 2018년 이후는 마지막에 평가합니다. 번들에 저장된 결과만 빠르게 열려면 `False`로 바꾸세요.


In [ ]:
RUN_SEARCH = True
if RUN_SEARCH:
    completed = subprocess.run(
        [sys.executable, "-u", "vkospi_robust_dynamic_experiment.py"],
        cwd=PROJECT_ROOT, text=True, capture_output=True, check=True,
    )
    print(completed.stdout[-6000:])
else:
    print("재탐색을 생략하고 번들의 검증 결과를 읽습니다.")


## 3. 상수·분류력·기간성과·오버피팅 진단

`RUN_DIAGNOSTICS=True`이면 배포 전략은 그대로 둔 채 다음 항목을 다시 계산합니다.

- 거시 상수 0.20·0.55·0.10·0.85의 1변수 민감도
- 16개 꼬리손실 설명변수의 AUC와 순차 로지스틱 계수
- ROC AUC·평균정밀도·Brier·LogLoss·ECE
- 요청한 2005~2026 구간의 데이터 가용성과 실제 전략 가능 구간
- 810개 격자의 승자 주변, 잠금 연도, 부트스트랩 기반 오버피팅 감사

이 진단은 잠금 결과를 보고 파라미터를 다시 고르는 데 쓰지 않습니다.


In [ ]:
RUN_DIAGNOSTICS = True
if RUN_DIAGNOSTICS:
    completed = subprocess.run(
        [sys.executable, "-u", "vkospi_extended_diagnostics.py"],
        cwd=PROJECT_ROOT, text=True, capture_output=True, check=True,
    )
    print(completed.stdout[-5000:])
else:
    print("재계산을 생략하고 번들에 저장된 진단 결과를 읽습니다.")


## 4. SJM·로지스틱 하이퍼파라미터 강건성

`RUN_MODEL_ROBUSTNESS=True`이면 다음 감사를 재실행합니다. 번들에는 1차 계산 캐시가 들어 있어 로지스틱 walk-forward는 재사용하고, SJM 후보 경로와 모든 요약·재현 단언을 다시 확인합니다.

- 로지스틱 28개: C, L1/L2/ElasticNet, liblinear/lbfgs/saga, 클래스 가중
- SJM 49개 시도: 점프 벌점, 희소 변수 수, 혼합비, SJM 미사용
- 2017년 이전 8블록 CSCV/PBO 70분할, 잠금 6개월 블록 부트스트랩 2,000회
- 2018년 이후는 재선정이 아니라 순위 안정성과 실패 확인에만 사용


In [ ]:
RUN_MODEL_ROBUSTNESS = True
if RUN_MODEL_ROBUSTNESS:
    completed = subprocess.run(
        [sys.executable, "-u", "vkospi_model_robustness.py"],
        cwd=PROJECT_ROOT, text=True, capture_output=True, check=True,
    )
    print(completed.stdout[-8000:])
else:
    print("재계산을 생략하고 번들의 강건성 결과를 읽습니다.")

RESULTS = PROJECT_ROOT / "results"
model_audit = json.loads((RESULTS / "vkospi_model_robustness.json").read_text(encoding="utf-8"))
display(Markdown("### 핵심 선택위험 — 낮을수록 안정적인 PBO"))
display(pd.Series({
    "로지스틱 예측 Brier PBO": model_audit["logistic"]["prelock_prediction_brier_pbo"]["pbo"],
    "로지스틱 포트폴리오 Sharpe PBO": model_audit["logistic"]["prelock_portfolio_sharpe_pbo"]["pbo"],
    "수렴경고 제외 Sharpe PBO": model_audit["logistic"]["prelock_portfolio_sharpe_pbo_excluding_warning_candidates"]["pbo"],
    "SJM 거시 Brier PBO": model_audit["sjm"]["prelock_macro_brier_pbo"]["pbo"],
    "SJM soft Sharpe PBO": model_audit["sjm"]["prelock_soft_sharpe_pbo"]["pbo"],
}).to_frame("PBO").style.format("{:.1%}"))

display(Markdown(
    "포트폴리오 기준 PBO가 높아 **오버피팅이 없다고 결론낼 수 없습니다.** "
    "잠금 결과는 더 좋아 보이는 후보로 갈아타는 데 사용하지 않습니다."
))


## 5. 승자와 잠금 성과

In [ ]:
RESULTS = PROJECT_ROOT / "results"
report = json.loads((RESULTS / "vkospi_robust_dynamic_validation.json").read_text(encoding="utf-8"))
display(Markdown("### 2017년 이전에 선택된 설정"))
display(pd.Series(report["winner"], name="value").to_frame())

locked = report["locked"]
metrics = pd.DataFrame({
    "Existing dynamic": pd.Series(locked["existing"]),
    "Robust dynamic": pd.Series(locked["robust"]),
    "Delta": pd.Series(locked["deltas"]),
}).loc[["CAGR", "Sharpe", "MDD", "Calmar"]]
display(Markdown("### Locked 2018–2026 · 월간 기준 재조정"))
display(metrics.style.format("{:.4f}"))
assert locked["passes_all_three"]
assert locked["deltas"]["CAGR"] > 0 and locked["deltas"]["Sharpe"] > 0 and locked["deltas"]["MDD"] >= 0
print("CAGR / Sharpe / MDD 동시 개선 검증 완료")


## 6. 누적 성과와 드로다운

In [ ]:
old = pd.read_csv(RESULTS / "vkospi_dynamic_reconciled_monthly.csv", index_col=0)
new = pd.read_csv(RESULTS / "vkospi_robust_dynamic_reconciled_monthly.csv", index_col=0)
common = old.index.intersection(new.index)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(common, old.loc[common, "nav"], color="#9ca6a1", lw=2, label="Existing dynamic")
axes[0].plot(common, new.loc[common, "nav"], color="#087f5b", lw=2.4, label="Robust dynamic")
axes[0].set_ylabel("NAV"); axes[0].grid(alpha=.2); axes[0].legend()
axes[1].plot(range(len(common)), 100 * old.loc[common, "drawdown"], color="#9ca6a1", lw=2, label="Existing")
axes[1].plot(range(len(common)), 100 * new.loc[common, "drawdown"], color="#087f5b", lw=2, label="Robust")
axes[1].set_ylabel("Drawdown (%)"); axes[1].grid(alpha=.2); axes[1].legend()
axes[1].set_xticks(range(0, len(common), 24), common[::24], rotation=45)
plt.tight_layout(); plt.show()


## 7. VKOSPI 가공 신호와 look-ahead 점검

In [ ]:
daily = pd.read_csv(RESULTS / "vkospi_robust_dynamic_daily.csv", parse_dates=["date", "signal_date"])
locked_daily = daily.loc[daily["date"] >= "2018-01-01"].copy()

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(locked_daily["date"], locked_daily["stress"], color="#d88934", lw=1)
axes[0].set_ylabel("Robust stress"); axes[0].grid(alpha=.2)
axes[1].fill_between(locked_daily["date"], 100 * locked_daily["transfer_fraction"], color="#087f5b", alpha=.55)
axes[1].set_ylabel("Transfer (%)"); axes[1].grid(alpha=.2)
plt.tight_layout(); plt.show()

valid = daily["signal_date"].notna()
assert (daily.loc[valid, "signal_date"].to_numpy() < daily.loc[valid, "date"].to_numpy()).all()
assert daily["stress"].between(0, 1).all()
print("모든 신호일 < 수익일, stress ∈ [0, 1]")


## 8. 기간·비용·불확실성

In [ ]:
comparison = pd.read_csv(RESULTS / "vkospi_robust_dynamic_comparison.csv")
costs = pd.read_csv(RESULTS / "vkospi_robust_dynamic_cost_sensitivity.csv")
display(comparison[["Period", "Strategy", "CAGR", "Sharpe", "MDD", "Calmar"]].style.format({"CAGR":"{:.2%}", "Sharpe":"{:.3f}", "MDD":"{:.2%}", "Calmar":"{:.3f}"}))
display(costs[["Period", "Strategy", "CAGR", "Sharpe", "MDD"]].style.format({"CAGR":"{:.2%}", "Sharpe":"{:.3f}", "MDD":"{:.2%}"}))

boot = locked["bootstrap"]
display(pd.Series({
    "CAGR 개선": boot["probability_cagr_improves"],
    "Sharpe 개선": boot["probability_sharpe_improves"],
    "MDD 개선": boot["probability_mdd_improves"],
    "세 지표 동시 개선": boot["probability_all_three_improve"],
}).to_frame("6개월 블록 부트스트랩").style.format("{:.1%}"))


## 9. AUC·Brier·16개 설명변수와 2005 요청구간

In [ ]:
tail_prediction = pd.read_csv(RESULTS / "vkospi_tail_prediction_diagnostics.csv")
tail_features = pd.read_csv(RESULTS / "vkospi_tail_feature_diagnostics.csv")
period_performance = pd.read_csv(RESULTS / "vkospi_extended_period_performance.csv")
macro_sensitivity = pd.read_csv(RESULTS / "vkospi_macro_constant_sensitivity.csv")
overfit = json.loads((RESULTS / "vkospi_overfitting_diagnostics.json").read_text(encoding="utf-8"))

display(Markdown("### 꼬리손실 분류 성능 — AUC는 순위, Brier는 확률오차"))
display(tail_prediction.style.format({
    "event_rate": "{:.2%}", "roc_auc": "{:.4f}", "average_precision": "{:.4f}",
    "brier_score": "{:.4f}", "calibration_prevalence_brier": "{:.4f}",
    "recall_at_top_20pct": "{:.2%}", "precision_at_top_20pct": "{:.2%}",
}))
display(Markdown(
    "잠금 AUC가 0.7565여도 Brier 0.2193은 단순 사건률 고정예측 0.0903보다 나쁩니다. "
    "그래서 원시 확률을 그대로 비중으로 쓰지 않고 과거 예측 내 인과적 백분위로 바꿉니다."
))

display(Markdown("### 16개 입력변수 — 단변량 AUC와 표준화 로지스틱 계수"))
display(tail_features.style.format({
    "raw_univariate_auc": "{:.3f}",
    "direction_free_auc": "{:.3f}",
    "median_standardized_logit_coefficient": "{:+.3f}",
    "coefficient_sign_stability": "{:.1%}",
}))

display(Markdown("### 2005~2026 요청구간과 실제 측정 가능 구간"))
display(period_performance.style.format({
    "CAGR": "{:.2%}", "Sharpe": "{:.3f}", "MDD": "{:.2%}", "Calmar": "{:.3f}",
}))
display(Markdown(
    "네 자산 공통 월수익은 2006-04에 시작하고 24개월 워밍업 뒤 첫 거래월은 2007-04입니다. "
    "따라서 2005년부터의 동일 전략 성과는 만들 수 없으며, 2005~2026 KODEX200은 별도 시장 벤치마크입니다."
))

display(Markdown("### 거시 상수 민감도와 오버피팅 감사"))
display(macro_sensitivity.loc[
    macro_sensitivity["parameter"].eq("deployed")
    | macro_sensitivity["parameter"].eq("sjm_weight")
].style.format({"mean_brier": "{:.4f}", "quadrant_accuracy": "{:.2%}"}))
display(pd.Series({
    "후보 수": overfit["candidate_count"],
    "엄격 통과": overfit["strict_pass_count"],
    "승자-차점자 점수차": overfit["winner_runner_score_gap"],
    "잠금 연도 우위": f'{overfit["locked_years_robust_outperformed"]}/{overfit["locked_years_total"]}',
    "결론": overfit["conclusion"],
}).to_frame("audit"))


## 10. Open Asset Pricing 연결과 한계

[Chen–Zimmermann SignalDoc Browser](https://openassetpricing.com/SignalDoc-Browser.html)는 개별 주식 횡단면 예측 신호 라이브러리입니다. 본 실험은 `betaVIX`, `RealizedVol`, momentum/trend 계열에서 **변동성 상태·방향·지속성을 과거 자료로 표현하는 원칙**만 참고해 VKOSPI 시장 시계열 입력으로 변환했습니다. 직접 복제나 같은 경제적 검정을 주장하지 않습니다.

관측 잠금 성과의 개선 폭은 작으며 부트스트랩의 세 지표 동시 개선 확률도 확정적이지 않습니다. 세금·슬리피지·추적오차·상품 교체와 미래 구조변화는 별도 검토해야 합니다.


## 11. 결과 다운로드

In [ ]:
output = Path("/content/vkospi_robust_dynamic_results.zip") if IN_COLAB else PROJECT_ROOT / "vkospi_robust_dynamic_results.zip"
with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    output_patterns = (
        "vkospi_robust_dynamic_*",
        "vkospi_macro_constant_sensitivity.csv",
        "vkospi_tail_feature_diagnostics.csv",
        "vkospi_tail_prediction_diagnostics.csv",
        "vkospi_extended_period_performance.csv",
        "vkospi_robust_grid_neighborhood.csv",
        "vkospi_overfitting_diagnostics.json",
        "vkospi_locked_annual_relative_performance.csv",
        "vkospi_logistic_*",
        "vkospi_sjm_*",
        "vkospi_model_robustness.json",
    )
    output_files = []
    for pattern in output_patterns:
        output_files.extend(RESULTS.glob(pattern))
    for path in sorted(set(output_files)):
        archive.write(path, arcname=path.name)
print("저장 완료:", output)
if IN_COLAB:
    from google.colab import files
    files.download(str(output))
